# Actividad 04: Esquema Estrella y Arquitectura del Data Warehouse

En esta actividad vas a diseñar un **esquema estrella** usando el enfoque de Kimball y luego lo implementarás a través de las **capas del Data Warehouse**: Source, Staging y Storage.

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect(':memory:')
print('Base de datos DuckDB en memoria creada.')

Base de datos DuckDB en memoria creada.


## Qué aprenderás

1. Aplicar los **4 pasos de Kimball** para diseñar un esquema estrella
2. Declarar el **grain** del modelo
3. Identificar **dimensiones** y **hechos**
4. Mover datos a través de las **capas del DW**

### El camino de Kimball

Primero el proceso y el grain, después las dimensiones y los hechos: así llegas al **esquema estrella**.

```mermaid
flowchart LR
    P["1. Proceso de negocio"] --> G["2. Grain"]
    G --> D["3. Dimensiones"]
    D --> F["4. Hechos y medidas"]
    F --> S["Esquema estrella"]
```


---

## El caso de negocio

Las preguntas de negocio que debe responder el warehouse están en el [brief de Wesker](../BRIEF.md).

Necesitas un **esquema estrella** con tablas de hechos y dimensiones capaz de responderlas.

---

## Paso 1: Identifica el proceso de negocio

> **Recuerda:** el proceso de negocio es el flujo operativo que generamos como hechos.

El proceso principal es: **experimentos realizados**.

Cada fila en la tabla de hechos representará **un experimento**.

---

## Paso 2: Declara el grain (nivel de detalle)

> **Recuerda:** el **grain** define qué representa exactamente una fila de la tabla de hechos: cuánto más fino, más preguntas responde.

El grain es: **un experimento por fila**.

Cada fila en `experiment_fact` representa un experimento único.

---

## Paso 3: Identifica las dimensiones

> **Recuerda:** las **dimensiones** aportan el contexto de los hechos (quién, qué, cuándo, dónde); los **hechos** son las medidas numéricas.

| Dimensión | Responde a | Columnas clave |
|-----------|------------|----------------|
| `date_dim` | **¿Cuándo?** | date_id, full_date, year, month, is_weekend |
| `laboratory_dim` | **¿Dónde?** | laboratory_id, laboratory_name, country |
| `researcher_dim` | **¿Quién?** | researcher_id, full_name, department |
| `sample_dim` | **¿Qué?** | sample_id, sample_type, risk_level |
| `project_dim` | **¿Qué proyecto?** | project_id, project_name, category |

---

## Paso 4: Identifica los hechos (medidas)

| Hecho | Tipo | Descripción |
|-------|------|-------------|
| `experiment_cost` | Aditivo | Coste total del experimento |
| `processing_time_minutes` | Aditivo | Tiempo de procesamiento |

### El esquema estrella que estás diseñando

La tabla de hechos en el centro y las dimensiones alrededor: cada fila de `experiment_fact` es **un experimento**, y las dimensiones responden a quién, dónde, cuándo y qué.

```mermaid
erDiagram
    DATE_DIM ||--o{ EXPERIMENT_FACT : FK-date
    LABORATORY_DIM ||--o{ EXPERIMENT_FACT : FK-lab
    RESEARCHER_DIM ||--o{ EXPERIMENT_FACT : FK-researcher
    SAMPLE_DIM ||--o{ EXPERIMENT_FACT : FK-sample
    PROJECT_DIM ||--o{ EXPERIMENT_FACT : FK-project
    EXPERIMENT_FACT {
        int experiment_id PK
        int date_id FK
        int laboratory_id FK
        int researcher_id FK
        int sample_id FK
        int project_id FK
        float experiment_cost
        int processing_time_minutes
    }
    DATE_DIM {
        int date_id PK
        date full_date
        int year
        int month
        bool is_weekend
    }
    LABORATORY_DIM {
        int laboratory_id PK
        string laboratory_name
        string country
    }
    RESEARCHER_DIM {
        int researcher_id PK
        string full_name
        string department
    }
    SAMPLE_DIM {
        int sample_id PK
        string sample_type
        string risk_level
    }
    PROJECT_DIM {
        int project_id PK
        string project_name
        string category
    }
```


---

## Arquitectura del DW: las 4 capas

> **Recuerda:** el DW se organiza en 4 capas: **Source** (datos crudos), **Staging** (extracción y limpieza), **Storage** (DW analítico) y **Presentation** (exposición a los usuarios).

En esta actividad movemos datos por las 3 primeras:

```
Source (CSV) -> Staging (tablas limpias) -> Storage (DW estrella)
```

| Capa | Qué hace | Ejemplo |
|------|----------|---------|
| **Source** | Datos crudos sin procesar | Archivos CSV |
| **Staging** | Limpieza y transformación | Tablas DuckDB |
| **Storage** | DW analítico | Esquema estrella |
| **Presentation** | Exposición a los usuarios | Las consultas analíticas de la actividad 05 |

El flujo es: **Source -> Staging -> Storage**; la capa **Presentation** la tocarás en la actividad 05.

### El flujo entre capas

En esta actividad recorres las 3 primeras capas. La cuarta (**Presentation**) es la que usarás para responder preguntas en la actividad 05.

```mermaid
flowchart LR
    SRC["Source<br/>CSV crudos"] --> STG["Staging<br/>tablas limpias"]
    STG --> STO["Storage<br/>DW estrella"]
    STO --> PRE["Presentation<br/>consultas de la actividad 05"]
```


---

## Parte 1: Source (datos crudos)

La carpeta `source/` contiene los datos crudos de Parasol Corp:
- `experimentos_raw.csv`: datos de experimentos
- `investigadores_raw.csv`: datos de investigadores
- `laboratorios_raw.csv`: datos de laboratorios

### Tarea 1.1: Explora los datos crudos

Lee los archivos CSV y muestra las primeras filas de cada uno.

In [2]:
from pathlib import Path

# Localizar los archivos desde el notebook
source_dir = Path("source")

# Cargar los tres CSV
df_experimentos = pd.read_csv(source_dir / "experimentos_raw.csv")
df_investigadores = pd.read_csv(source_dir / "investigadores_raw.csv")
df_laboratorios = pd.read_csv(source_dir / "laboratorios_raw.csv")

# Mostrar las primeras filas
display(df_experimentos.head())
display(df_investigadores.head())
display(df_laboratorios.head())

,experiment_id,researcher_name,lab_name,sample_type,cost_eur,experiment_date,status
0,1,Rebecca Chambers,Arklay,Blood,15000.0,2026-01-15,Completed
1,2,Marcus Aiken,Tanuki,Tissue,22000.0,2026-01-20,Completed
2,3,Ada Wong,Spencer,Saliva,18500.0,2026-02-01,Failed
3,4,Leon Kennedy,Arklay,Blood,9500.0,2026-02-10,Completed
4,5,Claire Redfield,Tanuki,DNA,31000.0,2026-02-15,In Progress


,researcher_id,full_name,department,position,hire_date,country
0,1,Rebecca Chambers,Bioresearch,Junior Scientist,2023-09-01,USA
1,2,Marcus Aiken,Bioresearch,Senior Scientist,2019-02-12,UK
2,3,Ada Wong,Advanced Research,Senior Scientist,2020-07-15,China
3,4,Leon Kennedy,Security,Field Analyst,2024-01-20,USA
4,5,Claire Redfield,Bioresearch,Junior Scientist,2023-11-05,USA


,lab_id,lab_name,city,country,security_level
0,1,Arklay,Ringtail City,USA,5
1,2,Tanuki,Osaka,Japan,3
2,3,Spencer,Geneva,Switzerland,4


### Tarea 1.2: Identifica problemas en los datos

Los datos crudos pueden tener problemas. Observa:
- Valores nulos
- Tipos de datos incorrectos
- Datos inconsistentes

In [3]:
# ============================================================
# TAREA 1.2 - ANÁLISIS DE CALIDAD DE DATOS
# ============================================================

dataframes = {
    "Experimentos": df_experimentos,
    "Investigadores": df_investigadores,
    "Laboratorios": df_laboratorios,
}

for nombre, df in dataframes.items():
    print(f"\n{'=' * 50}")
    print(f"DATASET: {nombre}")
    print(f"{'=' * 50}")

    # Dimensiones del dataset
    print(f"Registros: {df.shape[0]}")
    print(f"Columnas: {df.shape[1]}")

    # Tipos de datos y valores no nulos
    df.info()

    # Valores nulos por columna
    print("\nValores nulos:")
    print(df.isnull().sum())

    # Registros duplicados
    print(f"\nFilas duplicadas: {df.duplicated().sum()}")



DATASET: Experimentos
Registros: 10
Columnas: 7
<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   experiment_id    10 non-null     int64  
 1   researcher_name  10 non-null     str    
 2   lab_name         10 non-null     str    
 3   sample_type      10 non-null     str    
 4   cost_eur         10 non-null     float64
 5   experiment_date  10 non-null     str    
 6   status           10 non-null     str    
dtypes: float64(1), int64(1), str(5)
memory usage: 692.0 bytes

Valores nulos:
experiment_id      0
researcher_name    0
lab_name           0
sample_type        0
cost_eur           0
experiment_date    0
status             0
dtype: int64

Filas duplicadas: 0

DATASET: Investigadores
Registros: 5
Columnas: 6
<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype
---  -

---

## Parte 2: Staging (limpieza y transformación)

La carpeta `staging/` es donde cargamos los datos crudos a tablas de DuckDB y los limpiamos.

### Tarea 2.1: Carga los datos a tablas de staging

Crea tablas en DuckDB desde los CSV.

In [4]:
con.execute("""
    CREATE TABLE staging_experimentos AS
    SELECT *
    FROM read_csv_auto('source/experimentos_raw.csv');
""")

con.execute("""
    SELECT COUNT(*) AS total_experimentos
    FROM staging_experimentos;
""").df()

,total_experimentos
0,10


In [5]:
con.execute("""
    CREATE TABLE staging_investigadores AS
    SELECT *
    FROM read_csv_auto('source/investigadores_raw.csv');
""")

con.execute("""
    SELECT COUNT(*) AS total_investigadores
    FROM staging_investigadores;
""").df()


,total_investigadores
0,5


In [6]:
con.execute("""
    CREATE TABLE staging_laboratorios AS
    SELECT *
    FROM read_csv_auto('source/laboratorios_raw.csv');
""")

con.execute("""
    SELECT COUNT(*) AS total_laboratorios
    FROM staging_laboratorios;
""").df()


,total_laboratorios
0,3


### Tarea 2.2: Limpia los datos

Verifica que los datos están limpios:
- Sin valores nulos en columnas críticas
- Tipos de datos correctos

In [7]:
con.execute("""
    SELECT
        COUNT(*) AS total_registros,
        COUNT(*) FILTER (
            WHERE experiment_id IS NULL
        ) AS id_nulos,
        COUNT(*) FILTER (
            WHERE researcher_name IS NULL
        ) AS investigadores_nulos,
        COUNT(*) FILTER (
            WHERE lab_name IS NULL
        ) AS laboratorios_nulos,
        COUNT(*) FILTER (
            WHERE cost_eur IS NULL
        ) AS costes_nulos,
        COUNT(*) FILTER (
            WHERE experiment_date IS NULL
        ) AS fechas_nulas
    FROM staging_experimentos;
""").df()


,total_registros,id_nulos,investigadores_nulos,laboratorios_nulos,costes_nulos,fechas_nulas
0,10,0,0,0,0,0


In [8]:
con.execute("""
    SELECT
        e.experiment_id,
        e.researcher_name,
        e.lab_name
    FROM staging_experimentos AS e
    LEFT JOIN staging_investigadores AS i
        ON e.researcher_name = i.full_name
    LEFT JOIN staging_laboratorios AS l
        ON e.lab_name = l.lab_name
    WHERE i.researcher_id IS NULL
       OR l.lab_id IS NULL;
""").df()

,experiment_id,researcher_name,lab_name


In [9]:
con.execute("""
    DESCRIBE staging_experimentos;
""").df()

,column_name,column_type,null,key,default,extra
0,experiment_id,BIGINT,YES,None,None,None
1,researcher_name,VARCHAR,YES,None,None,None
2,lab_name,VARCHAR,YES,None,None,None
3,sample_type,VARCHAR,YES,None,None,None
4,cost_eur,DOUBLE,YES,None,None,None
5,experiment_date,DATE,YES,None,None,None
6,status,VARCHAR,YES,None,None,None


---

## Parte 3: Storage (Data Warehouse)

La carpeta `storage/` es donde vive el Data Warehouse con esquema estrella.

### Tarea 3.1: Crea las dimensiones del DW

Recuerda las dimensiones que identificamos en el Paso 3:
- `laboratory_dim`: donde ocurrio
- `researcher_dim`: quien lo realizo
- `date_dim`: cuando ocurrio

In [10]:
con.execute("""
    CREATE TABLE laboratory_dim (
        laboratory_id INTEGER PRIMARY KEY,
        laboratory_name VARCHAR NOT NULL,
        city VARCHAR,
        country VARCHAR,
        security_level INTEGER
    );
""")


In [11]:
con.execute("""
    INSERT INTO laboratory_dim (
        laboratory_id,
        laboratory_name,
        city,
        country,
        security_level
    )
    SELECT
        lab_id,
        lab_name,
        city,
        country,
        security_level
    FROM staging_laboratorios;
""")


In [12]:
con.execute("""
    SELECT *
    FROM laboratory_dim
    ORDER BY laboratory_id;
""").df()


,laboratory_id,laboratory_name,city,country,security_level
0,1,Arklay,Ringtail City,USA,5
1,2,Tanuki,Osaka,Japan,3
2,3,Spencer,Geneva,Switzerland,4


In [13]:
con.execute("""
    CREATE TABLE researcher_dim (
        researcher_id INTEGER PRIMARY KEY,
        researcher_name VARCHAR NOT NULL,
        department VARCHAR,
        position VARCHAR,
        hire_date DATE,
        country VARCHAR
    );
""")

In [14]:
con.execute("""
    INSERT INTO researcher_dim (
        researcher_id,
        researcher_name,
        department,
        position,
        hire_date,
        country
    )
    SELECT
        researcher_id,
        full_name,
        department,
        position,
        CAST(hire_date AS DATE),
        country
    FROM staging_investigadores;
""")

In [15]:
con.execute("""
    SELECT *
    FROM researcher_dim
    ORDER BY researcher_id;
""").df()

,researcher_id,researcher_name,department,position,hire_date,country
0,1,Rebecca Chambers,Bioresearch,Junior Scientist,2023-09-01,USA
1,2,Marcus Aiken,Bioresearch,Senior Scientist,2019-02-12,UK
2,3,Ada Wong,Advanced Research,Senior Scientist,2020-07-15,China
3,4,Leon Kennedy,Security,Field Analyst,2024-01-20,USA
4,5,Claire Redfield,Bioresearch,Junior Scientist,2023-11-05,USA


### Tarea 3.2: Crea la tabla de hechos

La tabla `experiment_fact` conecta todas las dimensiones.

Recuerda: cada fila representa **un experimento** (el grain que declaramos en el Paso 2).

In [ ]:
# TODO: Crea la tabla experiment_fact con FKs a las dimensiones
# Debe tener: experiment_id, laboratory_id, researcher_id, date_id, experiment_cost


### Tarea 3.3: Puebla el DW desde staging

Inserta los datos de staging a las tablas del DW.

### De staging al DW

Las tablas de staging alimentan a las dimensiones y la tabla de hechos se rellena cruzando staging con las dimensiones ya construidas.

```mermaid
flowchart TB
    SE[staging_experimentos] --> EF[experiment_fact]
    SE --> DD[date_dim]
    SL[staging_laboratorios] --> LD[laboratory_dim]
    SI[staging_investigadores] --> RD[researcher_dim]
    LD --> EF
    RD --> EF
    DD --> EF
```


In [16]:
con.execute("""
    CREATE TABLE date_dim (
        date_id INTEGER PRIMARY KEY,
        full_date DATE NOT NULL,
        day INTEGER,
        month INTEGER,
        quarter INTEGER,
        year INTEGER,
        month_name VARCHAR,
        day_of_week VARCHAR
    );
""")

In [17]:
con.execute("""
    INSERT INTO date_dim (
        date_id,
        full_date,
        day,
        month,
        quarter,
        year,
        month_name,
        day_of_week
    )
    SELECT DISTINCT
        CAST(STRFTIME(
            CAST(experiment_date AS DATE),
            '%Y%m%d'
        ) AS INTEGER) AS date_id,

        CAST(experiment_date AS DATE) AS full_date,

        DAY(CAST(experiment_date AS DATE)) AS day,
        MONTH(CAST(experiment_date AS DATE)) AS month,
        QUARTER(CAST(experiment_date AS DATE)) AS quarter,
        YEAR(CAST(experiment_date AS DATE)) AS year,

        STRFTIME(
            CAST(experiment_date AS DATE),
            '%B'
        ) AS month_name,

        STRFTIME(
            CAST(experiment_date AS DATE),
            '%A'
        ) AS day_of_week

    FROM staging_experimentos
    WHERE experiment_date IS NOT NULL;
""")

In [18]:
con.execute("""
    SELECT
        date_id,
        full_date,
        day,
        month,
        quarter,
        year
    FROM date_dim
    ORDER BY full_date;
""").df()

,date_id,full_date,day,month,quarter,year
0,20260115,2026-01-15,15,1,1,2026
1,20260120,2026-01-20,20,1,1,2026
2,20260201,2026-02-01,1,2,1,2026
3,20260210,2026-02-10,10,2,1,2026
4,20260215,2026-02-15,15,2,1,2026
5,20260301,2026-03-01,1,3,1,2026
6,20260305,2026-03-05,5,3,1,2026
7,20260310,2026-03-10,10,3,1,2026
8,20260315,2026-03-15,15,3,1,2026
9,20260320,2026-03-20,20,3,1,2026


In [19]:
con.execute("""
    SELECT
        year,
        quarter,
        month,
        COUNT(*) AS total_fechas
    FROM date_dim
    GROUP BY year, quarter, month
    ORDER BY year, quarter, month;
""").df()

,year,quarter,month,total_fechas
0,2026,1,1,2
1,2026,1,2,3
2,2026,1,3,5


In [20]:
con.execute("""
    CREATE TABLE experiment_fact (
        experiment_id INTEGER PRIMARY KEY,
        laboratory_id INTEGER NOT NULL,
        researcher_id INTEGER NOT NULL,
        date_id INTEGER NOT NULL,
        experiment_cost DECIMAL(12,2),
        sample_type VARCHAR,
        status VARCHAR,

        FOREIGN KEY (laboratory_id)
            REFERENCES laboratory_dim(laboratory_id),

        FOREIGN KEY (researcher_id)
            REFERENCES researcher_dim(researcher_id),

        FOREIGN KEY (date_id)
            REFERENCES date_dim(date_id)
    );
""")

In [21]:
con.execute("""
    INSERT INTO experiment_fact (
        experiment_id,
        laboratory_id,
        researcher_id,
        date_id,
        experiment_cost,
        sample_type,
        status
    )
    SELECT
        e.experiment_id,
        l.laboratory_id,
        r.researcher_id,

        CAST(
            STRFTIME(
                CAST(e.experiment_date AS DATE),
                '%Y%m%d'
            ) AS INTEGER
        ) AS date_id,

        CAST(e.cost_eur AS DECIMAL(12,2)),
        e.sample_type,
        e.status

    FROM staging_experimentos AS e

    INNER JOIN laboratory_dim AS l
        ON e.lab_name = l.laboratory_name

    INNER JOIN researcher_dim AS r
        ON e.researcher_name = r.researcher_name

    INNER JOIN date_dim AS d
        ON CAST(e.experiment_date AS DATE) = d.full_date;
""")

In [22]:
con.execute("""
    SELECT *
    FROM experiment_fact
    ORDER BY experiment_id;
""").df()

,experiment_id,laboratory_id,researcher_id,date_id,experiment_cost,sample_type,status
0,1,1,1,20260115,15000.0,Blood,Completed
1,2,2,2,20260120,22000.0,Tissue,Completed
2,3,3,3,20260201,18500.0,Saliva,Failed
3,4,1,4,20260210,9500.0,Blood,Completed
4,5,2,5,20260215,31000.0,DNA,In Progress
5,6,1,1,20260301,12000.0,Tissue,Completed
6,7,2,2,20260305,25000.0,Blood,Completed
7,8,3,3,20260310,19000.0,DNA,Completed
8,9,1,4,20260315,8500.0,Saliva,Failed
9,10,2,5,20260320,28000.0,Tissue,In Progress


In [23]:
con.execute("""
    SELECT
        COUNT(*) AS total_experimentos,
        SUM(experiment_cost) AS coste_total,
        AVG(experiment_cost) AS coste_medio
    FROM experiment_fact;
""").df()

,total_experimentos,coste_total,coste_medio
0,10,188500.0,18850.0


In [24]:
con.execute("""
    SELECT
        l.laboratory_name,
        l.country,
        COUNT(f.experiment_id) AS total_experimentos,
        SUM(f.experiment_cost) AS coste_total,
        ROUND(AVG(f.experiment_cost), 2) AS coste_medio
    FROM experiment_fact AS f
    INNER JOIN laboratory_dim AS l
        ON f.laboratory_id = l.laboratory_id
    GROUP BY
        l.laboratory_name,
        l.country
    ORDER BY coste_total DESC;
""").df()

,laboratory_name,country,total_experimentos,coste_total,coste_medio
0,Tanuki,Japan,4,106000.0,26500.0
1,Arklay,USA,4,45000.0,11250.0
2,Spencer,Switzerland,2,37500.0,18750.0


In [25]:
con.execute("""
    SELECT
        d.year,
        d.month,
        d.month_name,
        COUNT(f.experiment_id) AS total_experimentos,
        SUM(f.experiment_cost) AS coste_total
    FROM experiment_fact AS f
    INNER JOIN date_dim AS d
        ON f.date_id = d.date_id
    GROUP BY
        d.year,
        d.month,
        d.month_name
    ORDER BY d.year, d.month;
""").df()

,year,month,month_name,total_experimentos,coste_total
0,2026,1,January,2,37000.0
1,2026,2,February,3,59000.0
2,2026,3,March,5,92500.0


### Tarea 3.4: Consulta analítica

Verifica que tu DW responde a las preguntas de Wesker.

```sql
-- Cuanto cuesta cada laboratorio?
SELECT l.laboratory_name, SUM(f.experiment_cost) as total_cost
FROM experiment_fact f
JOIN laboratory_dim l ON f.laboratory_id = l.laboratory_id
GROUP BY l.laboratory_name;
```

In [27]:
con.execute("""
    SELECT
        COUNT(*) AS total_hechos,
        COUNT(l.laboratory_id) AS laboratorios_validos,
        COUNT(r.researcher_id) AS investigadores_validos,
        COUNT(d.date_id) AS fechas_validas
    FROM experiment_fact AS f
    LEFT JOIN laboratory_dim AS l
        ON f.laboratory_id = l.laboratory_id
    LEFT JOIN researcher_dim AS r
        ON f.researcher_id = r.researcher_id
    LEFT JOIN date_dim AS d
        ON f.date_id = d.date_id;
""").df()


,total_hechos,laboratorios_validos,investigadores_validos,fechas_validas
0,10,10,10,10


---

## Ampliación: el esquema snowflake

Tu estrella mantiene las dimensiones **desnormalizadas**: por ejemplo
`laboratory_dim` guarda el país dentro de la propia dimensión. Si **normalizas
una dimensión**, sacando sus atributos a tablas aparte, el esquema deja de ser
una estrella y pasa a ser un esquema **snowflake**: las dimensiones se abren en
ramas, como un copo de nieve.

Separando el país de `laboratory_dim` en una nueva `country_dim`:

```mermaid
erDiagram
    COUNTRY_DIM ||--o{ LABORATORY_DIM : FK-country
    LABORATORY_DIM ||--o{ EXPERIMENT_FACT : FK-lab
    COUNTRY_DIM {
        int country_id PK
        string country
    }
    LABORATORY_DIM {
        int laboratory_id PK
        string laboratory_name
        int country_id FK
    }
    EXPERIMENT_FACT {
        int experiment_id PK
        int laboratory_id FK
        float experiment_cost
    }
```

La única diferencia está en la dimensión: `LABORATORY_DIM` ya no guarda el país,
sino una `country_id` que apunta a la tabla nueva.

| | Estrella | Snowflake |
|---|---|---|
| Dimensiones | desnormalizadas | normalizadas |
| Consultas | menos JOINs | más JOINs |
| Espacio | algo de redundancia | menos redundancia |

En la práctica casi todos los DW usan estrella: la redundancia de las dimensiones
es pequeña y las consultas van más rápidas. Pero conviene reconocer un snowflake
cuando te lo encuentres.


---

## Resumen

En esta actividad has:

1. Aplicado los **4 pasos de Kimball** (proceso, grain, dimensiones, hechos).
2. Movido datos por las capas **Source -> Staging -> Storage**.
3. Construido el esquema estrella de Parasol Corp.

## Guarda tu trabajo

Copia tu solución en `scripts/star_schema_layers.py`.

```bash
pytest tests/ -v
```

### Siguiente: Actividad 05: Consultas analíticas